这一章对应 Java 后端八股里的 Spring MVC、IOC、AOP、事务、自动装配、MyBatis/JDBC 这些内容，但 Go 的工程化体系和 Java/Spring 很不一样。

Java 后端常见思路是：
- Spring MVC 负责 Web 请求分发。
- Spring IOC 负责 Bean 创建和依赖注入。
- Spring AOP 负责日志、事务、鉴权等横切逻辑。
- Spring 事务通过注解和代理自动管理事务边界。
- MyBatis 或 JPA 负责数据库访问。

Go 后端的思路更偏显式、组合、轻量：
- net/http 是标准库 Web 基础。
- Gin、Echo、Fiber 等框架是对 net/http 或其他底层网络库的封装。
- middleware 替代一部分 AOP 能力，用来处理日志、recover、鉴权、CORS、限流、trace。
- Go 没有 Spring 那种运行时大容器，依赖注入通常通过构造函数显式完成，也可以用 Wire 做编译期 DI。
- 数据库访问可以用 database/sql、sqlx、GORM，其中 GORM 更接近 ORM，database/sql 更底层、更可控。
- 事务通常要显式传递 *sql.Tx 或在 GORM 的 Transaction 回调中完成，不建议把事务边界藏得太深。

这一章要抓住一条主线：

Go Web 工程化不是把 Spring 原样搬过来，而是用更显式的方式组织 Web 请求、依赖、middleware、数据库连接池和事务边界。

面试里如果被问 Go Web，我一般会按这个顺序回答：

1. 先讲 net/http 的基本模型：Handler、Request、ResponseWriter、每请求 goroutine。
2. 再讲 Gin 在 net/http 上做了什么：路由树、Context、middleware、参数绑定、响应封装。
3. 再讲 middleware：它是 Go 里最常见的横切逻辑实现方式。
4. 再讲 DI：Go 没有 Spring IOC，主要通过构造函数注入和 interface 解耦。
5. 最后讲 GORM 和事务：连接池、Context、CRUD、Preload、事务边界、回滚、超时、不要混用 db 和 tx。


## 7.1 net/http 基础
### 7.1.1 Q：Go 的 net/http 是什么？
net/http 是 Go 标准库提供的 HTTP 服务端和客户端基础包。

在 Go 里，即使你使用 Gin、Echo、Kratos、go-zero 这类框架，底层也经常离不开 net/http 或类似的网络抽象。

net/http 的服务端核心有几个概念：
1. Handler
   这是最核心的接口。
```go
type Handler interface {
    ServeHTTP(ResponseWriter, *Request)
}
```

只要一个类型实现了 ServeHTTP 方法，它就可以处理 HTTP 请求。

2. HandlerFunc
   这是一个函数类型，它实现了 Handler 接口。
```go
type HandlerFunc func(ResponseWriter, *Request)
```
这让我们可以直接用普通函数处理请求。

3. Request
   表示客户端请求，包含 Method、URL、Header、Body、Context 等信息。

4. ResponseWriter
   用于向客户端写响应，包括状态码、响应头、响应体。

5. ServeMux
   标准库自带的路由分发器，根据 URL path 分发到不同 handler。

面试里可以这样回答：

Go 的 net/http 是标准库 HTTP 编程基础，它的核心抽象是 Handler 接口。一个请求进来后，Server 会把请求封装成 *http.Request，把响应写入抽象成 http.ResponseWriter，然后交给对应的 Handler 处理。Gin 这类框架本质上也是围绕请求路由、上下文封装和 middleware 链做增强。


In [ ]:
// 最简单的 net/http 服务
import (
    "fmt"
    "net/http"
)

func helloHandler(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintln(w, "hello net/http")
}

http.HandleFunc("/hello", helloHandler)
http.ListenAndServe(":8080", nil)

`http.HandleFunc("/hello", helloHandler)` 会把 `/hello` 路径绑定到 helloHandler。

当客户端访问 `/hello` 时，net/http 会创建 Request，提供 ResponseWriter，然后调用 `helloHandler(w, r)`

这里的 `w` 用来写响应，`r` 用来读请求。

> 在 notebook 里，这类长时间运行的 HTTP server 不适合直接阻塞执行。实际项目里一般写在 main 函数中。

## 【代码块 2】完整 main 版本

```go
package main

import (
    "fmt"
    "log"
    "net/http"
)

func helloHandler(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintln(w, "hello net/http")
}

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/hello", helloHandler)

    server := &http.Server{
        Addr:    ":8080",
        Handler: mux,
    }

    log.Println("server started at :8080")
    if err := server.ListenAndServe(); err != nil {
        log.Fatal(err)
    }
}
```

# 二、Handler 和 HandlerFunc

## 【文本块 4】Q：Handler 和 HandlerFunc 有什么区别？

Handler 是接口：

```go
type Handler interface {
    ServeHTTP(ResponseWriter, *Request)
}
```

HandlerFunc 是一个函数类型：

```go
type HandlerFunc func(ResponseWriter, *Request)
```

HandlerFunc 实现了 ServeHTTP 方法，所以普通函数可以被适配成 Handler。

这体现了 Go 的一个典型设计：

> 用小接口定义能力，用函数类型降低使用成本。

如果处理逻辑很简单，用 HandlerFunc 就够了。

如果处理器需要保存依赖，比如 service、logger、config，可以定义一个 struct 实现 Handler 接口。

---

## 【代码块 3】用 struct 实现 Handler

```go
package main

import (
    "fmt"
    "net/http"
)

type UserHandler struct {
    serviceName string
}

func (h *UserHandler) ServeHTTP(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintf(w, "handler from %s\n", h.serviceName)
}

func main() {
    mux := http.NewServeMux()

    userHandler := &UserHandler{
        serviceName: "user-service",
    }

    mux.Handle("/users", userHandler)

    http.ListenAndServe(":8080", mux)
}
```

---

## 【文本块 5】工程化理解

在真实项目里，Handler 往往不是裸函数，而是一个结构体。

例如：

```go
type UserHandler struct {
    userService *UserService
    logger      *zap.Logger
}
```

这样 handler 可以持有业务依赖。

这就和 Spring Controller 有一点类似，但区别在于：

Spring Controller 通常由 IOC 容器创建和注入依赖。
Go Handler 通常由 main 或 wire 生成代码显式创建。

Go 的风格更显式：

```go
repo := NewUserRepo(db)
svc := NewUserService(repo)
handler := NewUserHandler(svc)
```

这种写法看起来没有 Spring 自动装配那么“魔法”，但依赖关系更清楚，也更容易测试。

---

# 三、Request 和 ResponseWriter

## 【文本块 6】Q：http.Request 里有哪些重要字段？

http.Request 表示一次 HTTP 请求。

常用字段包括：

1. Method
   请求方法，比如 GET、POST、PUT、DELETE。

2. URL
   请求 URL，包括 Path、RawQuery 等。

3. Header
   请求头。

4. Body
   请求体，是 io.ReadCloser，只能读一次。

5. Context
   请求上下文，用于超时、取消、trace、用户信息传递等。

6. Form / PostForm
   表单参数，需要 ParseForm 或 ParseMultipartForm。

7. RemoteAddr
   客户端地址。

8. Host
   请求 Host。

工程里最重要的是 Body 和 Context。

Body 是流，读完就没了。
Context 会在客户端断开、请求超时、服务关闭时取消，下游 DB/RPC/Redis 调用应该尽量传递它。

---

## 【代码块 4】读取 Query 参数和 Header

```go
package main

import (
    "fmt"
    "net/http"
)

func queryHandler(w http.ResponseWriter, r *http.Request) {
    name := r.URL.Query().Get("name")
    token := r.Header.Get("Authorization")

    fmt.Fprintf(w, "name=%s, token=%s\n", name, token)
}

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/query", queryHandler)

    http.ListenAndServe(":8080", mux)
}
```

---

## 【代码块 5】读取 JSON Body

```go
package main

import (
    "encoding/json"
    "fmt"
    "net/http"
)

type CreateUserRequest struct {
    Name string `json:"name"`
    Age  int    `json:"age"`
}

func createUserHandler(w http.ResponseWriter, r *http.Request) {
    defer r.Body.Close()

    var req CreateUserRequest
    if err := json.NewDecoder(r.Body).Decode(&req); err != nil {
        http.Error(w, "invalid json", http.StatusBadRequest)
        return
    }

    fmt.Fprintf(w, "create user: name=%s age=%d\n", req.Name, req.Age)
}

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/users", createUserHandler)

    http.ListenAndServe(":8080", mux)
}
```

---

## 【文本块 7】追问：Request.Body 为什么只能读一次？

Request.Body 是一个流式 Reader，本质上表示客户端请求体的数据流。

读取 Body 时，数据从底层连接或缓冲中被消费掉。读完之后，下一次再读就没有数据了。

所以如果 middleware 读取了 Body，后面的 handler 就读不到了。

如果确实需要在 middleware 里读取 Body 并让后续继续读，需要把内容读出来后重新设置 Body：

```go
bodyBytes, _ := io.ReadAll(r.Body)
r.Body.Close()
r.Body = io.NopCloser(bytes.NewBuffer(bodyBytes))
```

但这会把请求体全部读入内存，大请求体要谨慎，可能造成内存压力。

---

## 【文本块 8】Q：ResponseWriter 使用有什么注意点？

ResponseWriter 用来写 HTTP 响应。

常见操作：

```go
w.Header().Set("Content-Type", "application/json")
w.WriteHeader(http.StatusOK)
w.Write([]byte("hello"))
```

注意点：

第一，WriteHeader 只能生效一次。
第一次调用 WriteHeader 后，状态码就确定了。

第二，如果没有显式调用 WriteHeader，第一次 Write 会默认写 200。

第三，先写 Header，再写状态码，再写 Body。
如果已经 Write 了 body，再设置 Header 通常就晚了。

第四，ResponseWriter 默认不容易拿到状态码。
如果 middleware 想记录响应状态码，需要自己包装 ResponseWriter。

---

## 【代码块 6】返回 JSON 响应

```go
package main

import (
    "encoding/json"
    "net/http"
)

type APIResponse struct {
    Code int    `json:"code"`
    Msg  string `json:"msg"`
    Data any    `json:"data,omitempty"`
}

func jsonHandler(w http.ResponseWriter, r *http.Request) {
    resp := APIResponse{
        Code: 0,
        Msg:  "success",
        Data: map[string]any{
            "name": "go",
        },
    }

    w.Header().Set("Content-Type", "application/json")
    w.WriteHeader(http.StatusOK)

    json.NewEncoder(w).Encode(resp)
}

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/json", jsonHandler)

    http.ListenAndServe(":8080", mux)
}
```

---

# 四、net/http 并发模型

## 【文本块 9】Q：net/http 是一个请求一个 goroutine 吗？

是的，从使用模型上看，Go 的 net/http server 会为每个连接和请求创建 goroutine 处理。

更准确地说：

* Listener 接收 TCP 连接。
* 每个连接会有对应 goroutine 负责读请求。
* 对于 HTTP/1.x，一个连接上的请求通常按顺序处理。
* 对于 HTTP/2，一个连接上可以并发多个 stream，处理模型会更复杂。
* 每个请求最终会进入对应 handler 执行业务逻辑。

这也是 Go Web 编程很自然的地方：

你可以像写同步代码一样写 handler。

```go
func handler(w http.ResponseWriter, r *http.Request) {
    user, err := service.GetUser(r.Context(), id)
}
```

如果 DB/RPC 阻塞，阻塞的是当前 goroutine，不是整个进程。

底层 runtime 会通过 GMP 调度和 netpoller 管理这些 goroutine。

面试里可以这样回答：

net/http 的编程模型基本是一个请求由 goroutine 处理。goroutine 很轻量，所以不需要像 Java BIO 那样担心一个连接一个 OS 线程。网络等待时，runtime 会挂起 goroutine，底层线程可以去执行其他 goroutine，这让 Go 可以用简单同步代码支撑较高并发。

---

## 【文本块 10】追问：goroutine 很轻量，是不是就不用控制并发？

不是。

goroutine 轻量不等于没有成本。

每个请求一个 goroutine 没问题，但如果请求里继续无限制启动 goroutine，就可能导致：

* goroutine 数量暴涨
* 内存上涨
* 下游 DB/RPC 被打爆
* 调度压力变大
* 请求延迟抖动
* goroutine 泄漏

所以工程里仍然要控制并发：

* DB 连接池限制
* Redis 连接池限制
* HTTP client 连接池限制
* worker pool
* semaphore
* rate limit
* context timeout
* 熔断降级

面试里可以补一句：

> Go 的并发便宜，但下游资源不便宜。真正需要控制的是资源并发度和请求生命周期。

---

# 五、http.Server 参数

## 【文本块 11】Q：生产环境里的 http.Server 要配置哪些参数？

很多示例代码喜欢写：

```go
http.ListenAndServe(":8080", nil)
```

这适合 demo，不适合生产。

生产里建议显式创建 http.Server，并配置超时参数。

常见参数：

1. ReadHeaderTimeout
   读取请求头的超时时间。非常重要，可以防慢速攻击。

2. ReadTimeout
   读取整个请求的超时时间，包括 body。

3. WriteTimeout
   写响应的超时时间。

4. IdleTimeout
   keep-alive 连接空闲超时时间。

5. MaxHeaderBytes
   最大请求头大小。

6. Handler
   路由或框架 handler。

生产写法一般是：

```go
srv := &http.Server{
    Addr:              ":8080",
    Handler:           router,
    ReadHeaderTimeout: 5 * time.Second,
    ReadTimeout:       10 * time.Second,
    WriteTimeout:      10 * time.Second,
    IdleTimeout:       60 * time.Second,
    MaxHeaderBytes:    1 << 20,
}
```

面试里可以这样回答：

生产环境不建议直接裸用 http.ListenAndServe，而应该显式配置 http.Server 的超时参数，尤其是 ReadHeaderTimeout、ReadTimeout、WriteTimeout、IdleTimeout，避免慢连接、慢请求或异常客户端拖垮服务。

---

## 【代码块 7】生产风格 http.Server

```go
package main

import (
    "fmt"
    "log"
    "net/http"
    "time"
)

func main() {
    mux := http.NewServeMux()

    mux.HandleFunc("/ping", func(w http.ResponseWriter, r *http.Request) {
        fmt.Fprintln(w, "pong")
    })

    srv := &http.Server{
        Addr:              ":8080",
        Handler:           mux,
        ReadHeaderTimeout: 5 * time.Second,
        ReadTimeout:       10 * time.Second,
        WriteTimeout:      10 * time.Second,
        IdleTimeout:       60 * time.Second,
        MaxHeaderBytes:    1 << 20,
    }

    log.Println("server started at :8080")
    if err := srv.ListenAndServe(); err != nil && err != http.ErrServerClosed {
        log.Fatal(err)
    }
}
```

---

# 六、优雅停机

## 【文本块 12】Q：Go HTTP 服务怎么优雅停机？

优雅停机的目标是：

* 不再接受新请求
* 已经进来的请求尽量处理完
* 超过一定时间还没处理完就强制退出
* 释放数据库、Redis、MQ 等资源

Go 标准库提供了：

```go
server.Shutdown(ctx)
```

Shutdown 会关闭监听器，不再接受新连接，然后等待已有连接处理完成。

通常配合 os.Signal 监听 SIGINT、SIGTERM。

流程：

1. 启动 HTTP server。
2. 监听系统信号。
3. 收到退出信号后创建带超时的 context。
4. 调用 server.Shutdown(ctx)。
5. 关闭数据库、Redis、MQ 等资源。
6. 程序退出。

面试里可以这样回答：

Go 服务优雅停机通常用 http.Server.Shutdown。收到 SIGTERM 后，先停止接受新请求，再等待存量请求完成，并设置一个最大等待时间。这样发布或扩容时不会直接中断正在处理的请求。

---

## 【代码块 8】优雅停机示例

```go
package main

import (
    "context"
    "fmt"
    "log"
    "net/http"
    "os"
    "os/signal"
    "syscall"
    "time"
)

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/ping", func(w http.ResponseWriter, r *http.Request) {
        time.Sleep(2 * time.Second)
        fmt.Fprintln(w, "pong")
    })

    srv := &http.Server{
        Addr:              ":8080",
        Handler:           mux,
        ReadHeaderTimeout: 5 * time.Second,
    }

    go func() {
        log.Println("server started at :8080")
        if err := srv.ListenAndServe(); err != nil && err != http.ErrServerClosed {
            log.Fatal(err)
        }
    }()

    quit := make(chan os.Signal, 1)
    signal.Notify(quit, syscall.SIGINT, syscall.SIGTERM)

    <-quit
    log.Println("shutting down server...")

    ctx, cancel := context.WithTimeout(context.Background(), 10*time.Second)
    defer cancel()

    if err := srv.Shutdown(ctx); err != nil {
        log.Println("server shutdown error:", err)
    }

    log.Println("server stopped")
}
```

---

## 【文本块 13】追问：Shutdown 和 Close 有什么区别？

Shutdown 是优雅关闭。

它会停止接受新连接，并等待已有请求处理完成，直到 context 超时。

Close 是立即关闭。

它会直接关闭监听器和连接，可能导致正在处理的请求中断。

生产环境通常优先 Shutdown。
只有在 Shutdown 超时或紧急退出时，才考虑 Close。

---

# 七、HTTP Client 工程化

## 【文本块 14】Q：Go HTTP Client 使用有什么坑？

Go 里发 HTTP 请求很简单：

```go
http.Get(url)
```

但生产环境不能无脑用默认 http.Client。

常见问题：

第一，没有超时。
默认 client 如果没有设置 Timeout，遇到网络卡住可能一直等待。

第二，没有连接池配置。
高并发场景需要配置 Transport 的连接池参数。

第三，没有正确关闭 Body。
响应体必须 Close，否则连接无法复用，可能导致连接泄漏。

第四，没有复用 client。
每次请求都 new 一个 client，会导致连接池无法复用。

第五，没有传 context。
无法跟随请求取消，也无法做链路超时控制。

生产建议：

* 全局复用 http.Client
* 设置 Timeout
* 配置 Transport
* 每次请求带 context
* 读取并关闭 response body
* 对外部请求设置重试、熔断、限流

---

## 【代码块 9】生产风格 HTTP Client

```go
package main

import (
    "context"
    "fmt"
    "io"
    "net"
    "net/http"
    "time"
)

var httpClient = &http.Client{
    Timeout: 5 * time.Second,
    Transport: &http.Transport{
        Proxy: http.ProxyFromEnvironment,
        DialContext: (&net.Dialer{
            Timeout:   3 * time.Second,
            KeepAlive: 30 * time.Second,
        }).DialContext,
        MaxIdleConns:          100,
        MaxIdleConnsPerHost:   20,
        IdleConnTimeout:       90 * time.Second,
        TLSHandshakeTimeout:   3 * time.Second,
        ResponseHeaderTimeout: 3 * time.Second,
    },
}

func fetch(ctx context.Context, url string) ([]byte, error) {
    req, err := http.NewRequestWithContext(ctx, http.MethodGet, url, nil)
    if err != nil {
        return nil, err
    }

    resp, err := httpClient.Do(req)
    if err != nil {
        return nil, err
    }
    defer resp.Body.Close()

    if resp.StatusCode >= 400 {
        return nil, fmt.Errorf("bad status: %d", resp.StatusCode)
    }

    return io.ReadAll(resp.Body)
}

func main() {
    ctx, cancel := context.WithTimeout(context.Background(), 2*time.Second)
    defer cancel()

    body, err := fetch(ctx, "https://example.com")
    if err != nil {
        fmt.Println("error:", err)
        return
    }

    fmt.Println(len(body))
}
```

---

## 【文本块 15】追问：为什么 response.Body 必须 Close？

Go 的 HTTP client 默认会复用 TCP 连接。

只有当 response body 被读完并关闭后，这个连接才可能被放回连接池复用。

如果不 Close：

* 连接无法复用
* 文件描述符泄漏
* 连接数上涨
* 最终可能导致请求卡住或报 too many open files

所以每次成功拿到 resp 后，都应该：

```go
defer resp.Body.Close()
```

如果为了连接复用更稳定，通常还要读完 body。
但如果 body 很大，不一定要 io.ReadAll，可以根据业务流式处理或丢弃。

---

# 八、Gin 基础

## 【文本块 16】Q：Gin 是什么？它和 net/http 什么关系？

Gin 是 Go 生态中非常常用的 Web 框架。

它主要提供：

* 高性能路由
* 分组路由
* middleware 机制
* 参数解析
* JSON 绑定
* 参数校验集成
* Context 封装
* 响应封装
* recover、logger 等默认中间件

Gin 的核心对象是：

```go
*gin.Engine
```

它本质上实现了 http.Handler，所以可以被 net/http.Server 使用。

也就是说：

```go
router := gin.Default()
server := &http.Server{
    Handler: router,
}
```

是很自然的写法。

面试里可以这样回答：

Gin 是对 net/http 的工程化封装。它实现了 http.Handler，在标准库基础上提供了更高效的路由树、Context、middleware 链、参数绑定和响应工具。它不是完全脱离 net/http 的另一个体系，而是站在 net/http 抽象之上。

---

## 【代码块 10】Gin 基本服务

```go
package main

import (
    "github.com/gin-gonic/gin"
)

func main() {
    r := gin.Default()

    r.GET("/ping", func(c *gin.Context) {
        c.JSON(200, gin.H{
            "message": "pong",
        })
    })

    r.Run(":8080")
}
```

---

## 【文本块 17】GoNB 注意事项

如果在 notebook 或本地项目中运行 Gin，需要先安装依赖：

```bash
go get github.com/gin-gonic/gin
```

在真实项目里，依赖应该写入 go.mod。

如果 GoNB 内核不方便直接启动长时间 HTTP server，可以把这类代码作为笔记或保存成 main.go 后运行。

---

# 九、Gin 路由与参数

## 【文本块 18】Q：Gin 常见参数获取方式有哪些？

Gin 里常见参数来源有四类：

第一，path 参数。

例如：

```text
/users/:id
```

获取：

```go
id := c.Param("id")
```

第二，query 参数。

例如：

```text
/users?page=1&page_size=20
```

获取：

```go
page := c.Query("page")
```

第三，header。

```go
token := c.GetHeader("Authorization")
```

第四，body JSON。

```go
var req CreateUserRequest
c.ShouldBindJSON(&req)
```

Gin 把这些能力都封装在 `*gin.Context` 上。

---

## 【代码块 11】Gin 获取参数

```go
package main

import (
    "net/http"

    "github.com/gin-gonic/gin"
)

type CreateUserRequest struct {
    Name string `json:"name" binding:"required"`
    Age  int    `json:"age" binding:"gte=0,lte=150"`
}

func main() {
    r := gin.Default()

    r.GET("/users/:id", func(c *gin.Context) {
        id := c.Param("id")
        verbose := c.Query("verbose")
        token := c.GetHeader("Authorization")

        c.JSON(http.StatusOK, gin.H{
            "id":      id,
            "verbose": verbose,
            "token":   token,
        })
    })

    r.POST("/users", func(c *gin.Context) {
        var req CreateUserRequest
        if err := c.ShouldBindJSON(&req); err != nil {
            c.JSON(http.StatusBadRequest, gin.H{
                "error": err.Error(),
            })
            return
        }

        c.JSON(http.StatusOK, gin.H{
            "name": req.Name,
            "age":  req.Age,
        })
    })

    r.Run(":8080")
}
```

---

## 【文本块 19】追问：BindJSON 和 ShouldBindJSON 有什么区别？

Gin 里常见有 BindJSON 和 ShouldBindJSON。

BindJSON 如果绑定失败，会自动写 400 响应，并且可能中断后续处理。
ShouldBindJSON 只返回 error，不会自动写响应。

工程里更推荐 ShouldBindJSON，因为它让错误处理更可控。

例如可以统一返回：

```json
{
  "code": "INVALID_ARGUMENT",
  "message": "参数错误"
}
```

而不是让框架自动返回默认格式。

---

# 十、Gin Context

## 【文本块 20】Q：gin.Context 是什么？

gin.Context 是 Gin 对一次 HTTP 请求上下文的封装。

它包含：

* 原始 *http.Request
* ResponseWriter
* path/query/header/body 参数获取方法
* JSON/XML/String 响应方法
* middleware 链控制
* key-value 存储
* 错误收集
* handler 执行状态

可以理解为 Gin 版的“请求上下文对象”。

但要注意：

gin.Context 不是标准库 context.Context。

标准库 context.Context 在：

```go
c.Request.Context()
```

里。

如果调用 DB/RPC/HTTP client，应该传递：

```go
ctx := c.Request.Context()
```

而不是把 `*gin.Context` 传到 service/repo 层。

面试里可以这样回答：

gin.Context 是 Gin 框架封装的请求上下文，主要服务于 handler 和 middleware；标准库 context.Context 用于超时、取消和跨边界传递。业务 service 和 repository 层应该接收 context.Context，而不是依赖 gin.Context，否则会把业务层和 Web 框架耦合死。

---

## 【代码块 12】从 Gin 传递标准 context 到 service

```go
package main

import (
    "context"
    "net/http"

    "github.com/gin-gonic/gin"
)

type UserService struct{}

func (s *UserService) GetUser(ctx context.Context, id string) (string, error) {
    // 这里后续可以把 ctx 传给 DB、Redis、RPC
    return "user-" + id, nil
}

func main() {
    svc := &UserService{}

    r := gin.Default()

    r.GET("/users/:id", func(c *gin.Context) {
        id := c.Param("id")

        user, err := svc.GetUser(c.Request.Context(), id)
        if err != nil {
            c.JSON(http.StatusInternalServerError, gin.H{"error": err.Error()})
            return
        }

        c.JSON(http.StatusOK, gin.H{"user": user})
    })

    r.Run(":8080")
}
```

---

## 【文本块 21】追问：gin.Context 能不能传到 goroutine 里？

要非常小心。

gin.Context 是和请求生命周期绑定的对象，Gin 内部可能会复用它。不要在请求结束后继续使用原始 `*gin.Context`。

如果确实要在 goroutine 里使用请求信息，应该先把需要的数据取出来，或者使用 `c.Copy()`。

但更推荐的做法是：

* 提前提取必要字段
* 使用标准 context 控制取消
* 不要在异步 goroutine 里继续写响应
* 不要把原始 gin.Context 长期持有

错误倾向：

```go
go func() {
    doSomething(c)
}()
```

更好的写法：

```go
userID := c.GetString("userID")
ctx := c.Request.Context()

go func(ctx context.Context, userID string) {
    doSomething(ctx, userID)
}(ctx, userID)
```

面试里可以说：

> gin.Context 不应该随意跨 goroutine 使用，尤其不能在请求结束后继续使用。异步任务应该复制必要数据，并用标准 context 管理生命周期。

---

# 十一、Middleware 基础

## 【文本块 22】Q：什么是 middleware？

middleware 可以理解为请求处理链中的中间层。

请求进入真正业务 handler 之前，可以先经过一系列 middleware：

* 日志
* recover
* trace_id
* 鉴权
* CORS
* 限流
* 熔断
* 请求体大小限制
* 参数预处理
* 统一错误处理
* metrics 统计

请求处理完成后，middleware 也可以继续做收尾逻辑：

* 记录响应状态码
* 记录耗时
* 上报指标
* 打印 access log

在 Java/Spring 里，这类横切逻辑可能通过 Filter、Interceptor、AOP 实现。
在 Go Web 里，最常见的实现就是 middleware。

面试里可以这样回答：

middleware 是 Go Web 中实现横切逻辑的主要方式。它通过包裹 handler，在请求前后插入统一逻辑，比如日志、recover、鉴权、限流、trace 和 metrics。它在 Go 里承担了一部分 Spring AOP 和 Servlet Filter 的角色。

---

## 【代码块 13】net/http middleware

```go
package main

import (
    "fmt"
    "log"
    "net/http"
    "time"
)

func LoggingMiddleware(next http.Handler) http.Handler {
    return http.HandlerFunc(func(w http.ResponseWriter, r *http.Request) {
        start := time.Now()

        next.ServeHTTP(w, r)

        log.Printf("method=%s path=%s cost=%s", r.Method, r.URL.Path, time.Since(start))
    })
}

func hello(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintln(w, "hello")
}

func main() {
    mux := http.NewServeMux()
    mux.HandleFunc("/hello", hello)

    handler := LoggingMiddleware(mux)

    http.ListenAndServe(":8080", handler)
}
```

---

## 【文本块 23】middleware 的本质

net/http middleware 的本质就是：

```go
func(next http.Handler) http.Handler
```

它接收一个 handler，返回一个新的 handler。

新的 handler 可以在调用 next 前后加入逻辑。

这是一种典型的装饰器模式。

```text
请求
 ↓
middleware A
 ↓
middleware B
 ↓
middleware C
 ↓
业务 handler
```

响应返回时则反向经过这些 middleware。

---

# 十二、Gin Middleware

## 【文本块 24】Q：Gin middleware 是怎么执行的？

Gin middleware 本质上也是 handler 链。

Gin 里的 middleware 类型是：

```go
func(c *gin.Context)
```

通过：

```go
r.Use(middleware1, middleware2)
```

注册。

在 middleware 内部，可以调用：

```go
c.Next()
```

表示继续执行后续 middleware 和 handler。

也可以调用：

```go
c.Abort()
```

表示中断后续处理。

典型流程：

```go
func Middleware() gin.HandlerFunc {
    return func(c *gin.Context) {
        // 请求前逻辑

        c.Next()

        // 请求后逻辑
    }
}
```

面试里可以这样回答：

Gin middleware 是一组按顺序执行的 handler。c.Next 会继续执行链上的后续 handler，后续执行完后再回到当前 middleware 做收尾。c.Abort 可以阻止后续 handler 执行，常用于鉴权失败、限流失败等场景。

---

## 【代码块 14】Gin 日志 middleware

```go
package main

import (
    "log"
    "time"

    "github.com/gin-gonic/gin"
)

func AccessLog() gin.HandlerFunc {
    return func(c *gin.Context) {
        start := time.Now()

        c.Next()

        cost := time.Since(start)
        status := c.Writer.Status()

        log.Printf("method=%s path=%s status=%d cost=%s",
            c.Request.Method,
            c.Request.URL.Path,
            status,
            cost,
        )
    }
}

func main() {
    r := gin.New()

    r.Use(AccessLog())
    r.Use(gin.Recovery())

    r.GET("/ping", func(c *gin.Context) {
        c.JSON(200, gin.H{"message": "pong"})
    })

    r.Run(":8080")
}
```

---

## 【代码块 15】Gin 鉴权 middleware

```go
package main

import (
    "net/http"

    "github.com/gin-gonic/gin"
)

func AuthMiddleware() gin.HandlerFunc {
    return func(c *gin.Context) {
        token := c.GetHeader("Authorization")
        if token == "" {
            c.JSON(http.StatusUnauthorized, gin.H{
                "error": "missing token",
            })
            c.Abort()
            return
        }

        // 假设解析 token 得到 userID
        c.Set("userID", "10001")

        c.Next()
    }
}

func main() {
    r := gin.Default()

    api := r.Group("/api")
    api.Use(AuthMiddleware())

    api.GET("/profile", func(c *gin.Context) {
        userID := c.GetString("userID")
        c.JSON(http.StatusOK, gin.H{
            "userID": userID,
        })
    })

    r.Run(":8080")
}
```

---

## 【文本块 25】追问：middleware 顺序重要吗？

非常重要。

一般推荐顺序：

1. Recover
   最外层兜底 panic，避免进程崩溃。

2. RequestID / TraceID
   尽早生成 trace_id，后续日志都能带上。

3. Access Log
   记录请求耗时和状态码。

4. CORS
   处理跨域。

5. Rate Limit
   限流，尽早挡掉过量请求。

6. Auth
   鉴权。

7. Business Handler
   业务处理。

8. Metrics
   可以和 log 一起记录结果。

不同团队顺序会略有差异，但核心原则是：

> 越通用、越底层、越需要兜底的 middleware 越靠外；越接近业务的 middleware 越靠内。

---

# 十三、Recover Middleware

## 【文本块 26】Q：Web 服务里为什么要有 recover middleware？

如果 handler 内部 panic，没有 recover，可能导致当前请求异常，甚至影响进程稳定性。

Gin 默认的 `gin.Default()` 会包含 Logger 和 Recovery。

但实际项目里通常会自定义 Recovery：

* 捕获 panic
* 打印堆栈
* 打印 trace_id、path、method、user_id
* 返回统一 500 响应
* 上报告警

recover middleware 不能把 panic 静默吞掉。
panic 往往意味着代码 bug 或未预期状态，必须记录和告警。

---

## 【代码块 16】Gin 自定义 recover middleware

```go
package main

import (
    "log"
    "net/http"
    "runtime/debug"

    "github.com/gin-gonic/gin"
)

func RecoveryMiddleware() gin.HandlerFunc {
    return func(c *gin.Context) {
        defer func() {
            if r := recover(); r != nil {
                log.Printf("panic: %v\nstack:\n%s", r, string(debug.Stack()))

                c.JSON(http.StatusInternalServerError, gin.H{
                    "code":    "INTERNAL_ERROR",
                    "message": "internal server error",
                })
                c.Abort()
            }
        }()

        c.Next()
    }
}

func main() {
    r := gin.New()
    r.Use(RecoveryMiddleware())

    r.GET("/panic", func(c *gin.Context) {
        panic("something wrong")
    })

    r.Run(":8080")
}
```

---

## 【文本块 27】追问：recover middleware 能捕获 goroutine 里的 panic 吗？

不能。

recover 只能捕获当前 goroutine 的 panic。

如果 handler 里启动了新的 goroutine：

```go
go func() {
    panic("boom")
}()
```

外层 middleware 捕获不到。

所以任何自己启动的 goroutine，都应该在 goroutine 入口单独 recover。

工程里可以封装一个 GoSafe：

```go
func GoSafe(fn func()) {
    go func() {
        defer func() {
            if r := recover(); r != nil {
                log.Printf("panic: %v", r)
            }
        }()
        fn()
    }()
}
```

---

# 十四、DI：Go 里的依赖注入

## 【文本块 28】Q：Go 没有 Spring IOC，怎么做依赖注入？

Go 里通常不用 Spring 那种运行时 IOC 容器。

最常见方式是构造函数注入。

例如：

```go
repo := NewUserRepo(db)
svc := NewUserService(repo)
handler := NewUserHandler(svc)
```

每一层的依赖都通过构造函数显式传入。

典型分层：

* Handler：负责 HTTP 参数解析和响应
* Service：负责编排业务逻辑
* Repository/DAO：负责数据访问
* Client：负责调用外部服务
* Config：配置
* Logger：日志
* DB/Redis/MQ：基础设施

构造函数注入的优点：

1. 依赖关系清晰
2. 编译期可检查
3. 不依赖反射
4. 方便单元测试
5. 启动过程可控
6. 不容易出现隐藏依赖

缺点是：

1. main 里初始化代码比较长
2. 大项目依赖多时手写 wire-up 比较繁琐

所以大型项目里可以使用 Wire 做编译期依赖注入。

面试里可以这样回答：

Go 里依赖注入通常通过构造函数显式完成，而不是运行时 IOC 容器。这样依赖关系清楚、类型安全、方便测试。项目复杂后可以用 Google Wire 做编译期 DI，生成初始化代码，但本质仍然是显式构造和传参。

---

## 【代码块 17】构造函数注入示例

```go
package main

import (
    "context"
    "fmt"
)

type User struct {
    ID   int64
    Name string
}

type UserRepo interface {
    FindByID(ctx context.Context, id int64) (*User, error)
}

type userRepo struct{}

func NewUserRepo() UserRepo {
    return &userRepo{}
}

func (r *userRepo) FindByID(ctx context.Context, id int64) (*User, error) {
    return &User{
        ID:   id,
        Name: "Tom",
    }, nil
}

type UserService struct {
    repo UserRepo
}

func NewUserService(repo UserRepo) *UserService {
    return &UserService{
        repo: repo,
    }
}

func (s *UserService) GetUser(ctx context.Context, id int64) (*User, error) {
    return s.repo.FindByID(ctx, id)
}

func main() {
    repo := NewUserRepo()
    svc := NewUserService(repo)

    user, _ := svc.GetUser(context.Background(), 1)
    fmt.Println(user)
}
```

---

## 【文本块 29】为什么要依赖 interface，而不是具体实现？

这里有一个 Go 工程化细节：

Service 层依赖的是：

```go
type UserRepo interface {
    FindByID(ctx context.Context, id int64) (*User, error)
}
```

而不是具体的 `*userRepo`。

好处是：

1. 方便单元测试时注入 mock
2. 降低 service 对数据库实现的耦合
3. 后续可以替换 MySQL、Redis、RPC 实现
4. 符合面向接口编程

但 Go 里也不要滥用 interface。

如果只有一个实现、也没有测试替换需求，过早抽接口会增加复杂度。

常见经验是：

> interface 放在使用方，而不是实现方。

也就是说，谁需要这个能力，谁定义最小接口。

Service 只需要 repo 的 FindByID，就定义这个最小接口，而不是依赖一个巨大 Repository 接口。

---

## 【文本块 30】Go DI 和 Spring IOC 的区别

Spring IOC 的特点：

* 容器负责创建 Bean
* 通过注解或 XML 描述依赖
* 运行时反射和代理较多
* 自动装配
* 生命周期由容器管理

Go 构造函数注入的特点：

* main 或 wire 负责组装对象
* 依赖通过构造函数显式传递
* 编译期类型检查
* 少反射、少魔法
* 生命周期更直接

面试里可以这样对比：

Spring IOC 更像一个运行时容器，适合大型 Java 生态统一管理 Bean；Go 更强调显式依赖和组合，通常通过构造函数注入完成依赖管理。Go 的方式样板代码多一点，但依赖关系清晰，可测试性好，也更符合 Go 简单显式的风格。

---

# 十五、Wire、Fx、Dig

## 【文本块 31】Q：Go 里常见 DI 工具有哪些？

常见有三类：

第一，手写构造函数。
最简单，最推荐作为默认方案。

第二，Google Wire。
编译期依赖注入。它根据 provider 函数生成初始化代码，没有运行时反射。

优点：

* 编译期检查
* 运行时无额外开销
* 生成代码可读
* 符合 Go 显式风格

缺点：

* 需要生成代码
* 学习成本略高

第三，Uber Fx / Dig。
运行时依赖注入容器，功能更像 Spring，但使用反射。

优点：

* 适合复杂生命周期管理
* 模块化能力强
* 大型服务初始化方便

缺点：

* 运行时反射
* 依赖关系不如手写直观
* 出错可能在启动时才暴露
* 对 Go 新手不够透明

面试里可以这样回答：

小项目手写构造函数就够了。中大型项目可以用 Wire 做编译期 DI，保持类型安全和显式初始化。Fx/Dig 更像运行时容器，适合复杂模块和生命周期管理，但会引入反射和框架心智成本。

---

# 十六、项目分层

## 【文本块 32】Q：Go Web 项目一般怎么分层？

常见分层方式：

```text
cmd/
  server/
    main.go

internal/
  config/
  handler/
  service/
  repository/
  middleware/
  model/
  client/
  pkg/
```

或者：

```text
internal/
  user/
    handler.go
    service.go
    repository.go
    model.go
  order/
    handler.go
    service.go
    repository.go
    model.go
```

第一种是按技术分层。
第二种是按业务模块分层。

两种都可以。项目小的时候按技术分层更直观；项目大了以后，按业务模块聚合通常更容易维护。

一个典型请求链路：

```text
HTTP Request
   ↓
Middleware
   ↓
Handler
   ↓
Service
   ↓
Repository
   ↓
Database
```

每层职责：

Handler：

* 解析 HTTP 参数
* 参数校验
* 调用 service
* 转换 HTTP 响应

Service：

* 编排业务流程
* 处理事务边界
* 调用 repo/client
* 业务规则判断

Repository：

* 封装数据库访问
* 处理 SQL/GORM
* 不写复杂业务逻辑

Model：

* 数据结构定义
* Entity / DTO / PO 分离或适度合并

Middleware：

* 日志
* recover
* 鉴权
* trace
* 限流

面试里可以这样回答：

Go Web 项目通常会把 HTTP 层、业务层、数据访问层分开。Handler 不直接写数据库逻辑，Service 不依赖 Gin，Repository 不关心 HTTP。这样可以避免框架污染业务层，也方便单元测试和后续迁移。

---

# 十七、GORM 基础

## 【文本块 33】Q：GORM 是什么？和 database/sql 有什么区别？

GORM 是 Go 生态里常用的 ORM 框架。

它提供：

* struct 和表映射
* CRUD 封装
* 自动迁移
* 关联查询
* 事务
* hook
* soft delete
* preload
* scope
* logger
* plugin

database/sql 是标准库提供的数据库访问基础接口。它更底层、更通用，负责连接池、查询、事务等基础能力，但不提供 ORM 映射。

区别：

database/sql：

* 更轻量
* 更可控
* SQL 显式
* 性能和行为更透明
* 复杂业务需要手写较多代码

GORM：

* 开发效率高
* CRUD 简单
* struct 映射方便
* 关联处理方便
* 但生成 SQL 需要关注
* 复杂查询可能不如手写 SQL 清晰
* 容易出现 N+1、误更新、字段零值更新等坑

面试里可以这样回答：

database/sql 是 Go 标准库的底层数据库接口，GORM 是基于它之上的 ORM。GORM 提升了开发效率，适合常规 CRUD 和模型映射；但复杂查询和性能敏感路径，我会更倾向于手写 SQL 或至少打印并审查 GORM 生成的 SQL。

---

## 【代码块 18】GORM 基本连接

```go
package main

import (
    "fmt"
    "log"

    "gorm.io/driver/mysql"
    "gorm.io/gorm"
)

type User struct {
    ID   uint
    Name string
    Age  int
}

func main() {
    dsn := "user:password@tcp(127.0.0.1:3306)/test?charset=utf8mb4&parseTime=True&loc=Local"

    db, err := gorm.Open(mysql.Open(dsn), &gorm.Config{})
    if err != nil {
        log.Fatal(err)
    }

    fmt.Println(db)
}
```

---

## 【文本块 34】GoNB 注意事项

GORM 需要依赖：

```bash
go get gorm.io/gorm
go get gorm.io/driver/mysql
```

如果没有本地 MySQL，这段代码可以作为笔记理解，不一定在 notebook 里直接运行。

---

# 十八、GORM 连接池

## 【文本块 35】Q：GORM 怎么配置连接池？

GORM 底层使用 database/sql。

通过：

```go
sqlDB, err := db.DB()
```

可以拿到底层 `*sql.DB`，然后配置连接池。

常用参数：

1. SetMaxOpenConns
   最大打开连接数。

2. SetMaxIdleConns
   最大空闲连接数。

3. SetConnMaxLifetime
   连接最大生命周期。

4. SetConnMaxIdleTime
   连接最大空闲时间。

这些参数非常重要。

如果 MaxOpenConns 太小，高并发时请求会排队等待数据库连接。
如果太大，可能把数据库打爆。
如果 ConnMaxLifetime 不设置，可能遇到 MySQL 服务端连接回收导致坏连接问题。
如果 Idle 太多，会浪费连接资源。

---

## 【代码块 19】GORM 配置连接池

```go
package main

import (
    "log"
    "time"

    "gorm.io/driver/mysql"
    "gorm.io/gorm"
)

func main() {
    dsn := "user:password@tcp(127.0.0.1:3306)/test?charset=utf8mb4&parseTime=True&loc=Local"

    db, err := gorm.Open(mysql.Open(dsn), &gorm.Config{})
    if err != nil {
        log.Fatal(err)
    }

    sqlDB, err := db.DB()
    if err != nil {
        log.Fatal(err)
    }

    sqlDB.SetMaxOpenConns(100)
    sqlDB.SetMaxIdleConns(20)
    sqlDB.SetConnMaxLifetime(30 * time.Minute)
    sqlDB.SetConnMaxIdleTime(10 * time.Minute)
}
```

---

## 【文本块 36】追问：sql.DB 是一个连接吗？

不是。

这是 Go 数据库面试里非常常见的坑。

`*sql.DB` 不是单个数据库连接，而是一个连接池句柄。

它内部会按需创建、复用、回收连接。
它是并发安全的，应该作为全局对象复用，而不是每次请求都创建。

错误做法：

```go
func handler() {
    db, _ := sql.Open(...)
    defer db.Close()
}
```

这会导致每个请求创建连接池，非常糟糕。

正确做法：

服务启动时初始化一次 DB，在整个进程内复用。

面试里可以这样回答：

`*sql.DB` 是连接池，不是单个连接。GORM 底层也是通过 database/sql 管理连接池。生产环境需要配置最大连接数、空闲连接数、连接生命周期，并在服务启动时初始化一次，全局复用。

---

# 十九、GORM Model 和 Tag

## 【文本块 37】Q：GORM model 怎么定义？

GORM 通过 struct 映射数据库表。

常见写法：

```go
type User struct {
    ID        uint      `gorm:"primaryKey"`
    Name      string    `gorm:"size:64;not null"`
    Email     string    `gorm:"uniqueIndex;size:128"`
    CreatedAt time.Time
    UpdatedAt time.Time
    DeletedAt gorm.DeletedAt `gorm:"index"`
}
```

常见 tag：

* primaryKey
* column
* type
* size
* not null
* default
* uniqueIndex
* index
* autoCreateTime
* autoUpdateTime
* embedded
* foreignKey
* references

GORM 默认约定：

* struct 名字转复数蛇形表名
* 字段名转蛇形列名
* ID 是主键
* CreatedAt、UpdatedAt 自动维护
* DeletedAt 表示软删除

面试里可以这样回答：

GORM 通过 struct tag 描述字段和数据库列的映射。它有一套默认约定，比如 ID 主键、CreatedAt/UpdatedAt 自动维护、表名字段名蛇形转换。简单场景可以依赖约定，复杂场景要显式写 column、index、type 等 tag，避免生成 SQL 和表结构不符合预期。

---

## 【代码块 20】GORM Model 示例

```go
package main

import (
    "time"

    "gorm.io/gorm"
)

type User struct {
    ID        uint           `gorm:"primaryKey"`
    Name      string         `gorm:"size:64;not null"`
    Email     string         `gorm:"size:128;uniqueIndex"`
    Age       int            `gorm:"default:0"`
    CreatedAt time.Time
    UpdatedAt time.Time
    DeletedAt gorm.DeletedAt `gorm:"index"`
}
```

---

## 【文本块 38】追问：AutoMigrate 能不能直接在线上用？

要谨慎。

AutoMigrate 适合开发环境或简单项目，它会根据 model 自动创建或调整表结构。

但在线上生产环境，数据库 schema 变更通常要走 migration 流程：

* 评估影响
* 审核 SQL
* 灰度执行
* 大表变更避免锁表
* 可回滚方案
* 与代码发布顺序配合

AutoMigrate 可能生成不符合预期的 DDL，也不适合复杂变更。

面试里可以这样回答：

开发环境可以用 AutoMigrate 提高效率，但生产环境我更倾向于使用显式 migration 工具和审核过的 SQL。数据库表结构变更是高风险操作，不应该完全交给 ORM 自动决定。

---

# 二十、GORM CRUD

## 【文本块 39】Q：GORM 常见 CRUD 怎么写？

常见操作：

Create：

```go
db.Create(&user)
```

First：

```go
db.First(&user, id)
```

Where：

```go
db.Where("email = ?", email).First(&user)
```

Find：

```go
db.Where("age > ?", 18).Find(&users)
```

Updates：

```go
db.Model(&user).Updates(map[string]any{"name": "Tom"})
```

Delete：

```go
db.Delete(&user)
```

注意：

* First 找不到会返回 ErrRecordNotFound
* Find 找不到通常不返回 ErrRecordNotFound，而是返回空 slice
* Updates struct 时默认不会更新零值字段
* 想更新零值可以用 map 或 Select
* Delete 如果有 DeletedAt，默认软删除

---

## 【代码块 21】GORM CRUD 示例

```go
package main

import (
    "errors"
    "fmt"

    "gorm.io/gorm"
)

type User struct {
    ID    uint
    Name  string
    Email string
    Age   int
}

func createUser(db *gorm.DB) error {
    user := User{Name: "Tom", Email: "tom@example.com", Age: 18}
    return db.Create(&user).Error
}

func findUser(db *gorm.DB, id uint) (*User, error) {
    var user User

    err := db.First(&user, id).Error
    if errors.Is(err, gorm.ErrRecordNotFound) {
        return nil, nil
    }
    if err != nil {
        return nil, err
    }

    return &user, nil
}

func updateUser(db *gorm.DB, id uint) error {
    return db.Model(&User{}).
        Where("id = ?", id).
        Updates(map[string]any{
            "name": "Jerry",
            "age":  0,
        }).Error
}

func deleteUser(db *gorm.DB, id uint) error {
    return db.Delete(&User{}, id).Error
}

func demo(db *gorm.DB) {
    user, err := findUser(db, 1)
    fmt.Println(user, err)
}
```

---

## 【文本块 40】追问：GORM Updates struct 为什么不更新零值？

GORM 用 struct 做 Updates 时，默认只更新非零值字段。

例如：

```go
db.Model(&user).Updates(User{
    Name: "",
    Age:  0,
})
```

Name 和 Age 可能不会被更新，因为它们是零值。

如果确实要更新零值，有几种方式：

第一，用 map。

```go
db.Model(&user).Updates(map[string]any{
    "name": "",
    "age":  0,
})
```

第二，用 Select。

```go
db.Model(&user).Select("name", "age").Updates(User{
    Name: "",
    Age:  0,
})
```

第三，单字段 Update。

```go
db.Model(&user).Update("age", 0)
```

面试里可以这样回答：

GORM struct Updates 默认忽略零值字段，这是为了避免误更新。但这也容易踩坑。如果业务需要把字段更新为空字符串、0、false，应该用 map、Select 或 Update 明确指定字段。

---

# 二十一、GORM 查询优化

## 【文本块 41】Q：GORM 容易有哪些性能问题？

常见问题有：

第一，N+1 查询。
查询列表后，又循环查询每个对象的关联数据。

第二，查询字段过多。
没有 Select，只是 `Find(&users)`，把不需要的大字段也查出来。

第三，没有索引。
ORM 写法隐藏了 SQL，容易忽略 explain。

第四，Preload 滥用。
一次加载大量关联对象，导致内存和 SQL 压力很大。

第五，分页深翻页。
offset 很大时性能差。

第六，更新误操作。
Where 条件缺失或过宽。

第七，软删除影响查询。
GORM 默认会过滤 DeletedAt，但复杂 SQL 要注意。

第八，日志不清楚。
没有打开慢 SQL 日志，问题定位困难。

面试里可以这样回答：

GORM 提高了开发效率，但不能替代 SQL 思维。性能敏感场景要关注生成的 SQL、索引、字段选择、N+1、Preload 范围、分页方式和慢 SQL。复杂查询我更倾向手写 SQL 或至少用 Debug/Logger 审查 SQL。

---

## 【代码块 22】Select 减少查询字段

```go
var users []User

err := db.Select("id", "name", "email").
    Where("age >= ?", 18).
    Find(&users).Error
```

---

## 【代码块 23】分页查询

```go
func ListUsers(db *gorm.DB, page, pageSize int) ([]User, error) {
    if page <= 0 {
        page = 1
    }
    if pageSize <= 0 || pageSize > 100 {
        pageSize = 20
    }

    offset := (page - 1) * pageSize

    var users []User
    err := db.Where("age >= ?", 18).
        Order("id DESC").
        Offset(offset).
        Limit(pageSize).
        Find(&users).Error

    return users, err
}
```

---

## 【文本块 42】追问：深分页怎么优化？

传统分页：

```sql
limit 20 offset 1000000
```

offset 很大时，数据库需要扫描并丢弃大量数据，性能会很差。

优化方式：

第一，基于游标分页。

```sql
where id < last_id order by id desc limit 20
```

第二，覆盖索引先查 id，再回表。

第三，限制最大页数。

第四，搜索场景使用专门搜索引擎。

在 Go/GORM 中也可以写：

```go
db.Where("id < ?", lastID).
    Order("id DESC").
    Limit(pageSize).
    Find(&users)
```

面试里可以这样回答：

深分页不要无脑 offset，大 offset 会让数据库扫描大量无用行。更推荐基于游标的分页，比如用 id 或 created_at 作为游标，where id < last_id order by id desc limit n。

---

# 二十二、Preload 和 Joins

## 【文本块 43】Q：GORM 的 Preload 和 Joins 有什么区别？

Preload 是预加载关联数据，通常会发多条 SQL。

例如：

```go
db.Preload("Orders").Find(&users)
```

大致会执行：

```sql
select * from users;
select * from orders where user_id in (...);
```

Joins 是连接查询，通常通过 SQL JOIN 一次查出关联数据。

例如：

```go
db.Joins("Company").Find(&users)
```

区别：

Preload：

* 多条 SQL
* 避免笛卡尔积膨胀
* 适合一对多关联
* 更容易映射到结构体

Joins：

* 一条 SQL
* 适合一对一或多对一
* 可以按关联表字段过滤排序
* 一对多时可能导致重复行

面试里可以这样回答：

Preload 更像批量关联查询，通常适合一对多；Joins 是 SQL 连接查询，适合需要按关联表过滤、排序或一对一关系。Preload 可以避免 N+1，但也要控制关联数据量，不能无限预加载。

---

## 【代码块 24】Preload 示例

```go
type User struct {
    ID     uint
    Name   string
    Orders []Order
}

type Order struct {
    ID     uint
    UserID uint
    Amount int64
}

func ListUsersWithOrders(db *gorm.DB) ([]User, error) {
    var users []User

    err := db.Preload("Orders").
        Limit(20).
        Find(&users).Error

    return users, err
}
```

---

# 二十三、GORM Context

## 【文本块 44】Q：GORM 怎么传 context？

GORM 支持：

```go
db.WithContext(ctx)
```

这样数据库操作就可以感知 context 的取消和超时。

在 Web 请求中，应该把请求的 context 传到 service，再传到 repository。

例如：

```go
func (r *UserRepo) FindByID(ctx context.Context, id uint) (*User, error) {
    err := r.db.WithContext(ctx).First(&user, id).Error
}
```

这样如果客户端断开、请求超时、服务关闭，下游 DB 查询也有机会取消。

面试里可以这样回答：

GORM 操作应该使用 WithContext(ctx)，并且 ctx 应该从 HTTP 请求一路传到 repository。这样可以统一控制超时和取消，避免请求已经结束但数据库查询还在跑。

---

## 【代码块 25】Repository 中使用 GORM WithContext

```go
package repository

import (
    "context"
    "errors"

    "gorm.io/gorm"
)

type User struct {
    ID   uint
    Name string
}

type UserRepo struct {
    db *gorm.DB
}

func NewUserRepo(db *gorm.DB) *UserRepo {
    return &UserRepo{db: db}
}

func (r *UserRepo) FindByID(ctx context.Context, id uint) (*User, error) {
    var user User

    err := r.db.WithContext(ctx).First(&user, id).Error
    if errors.Is(err, gorm.ErrRecordNotFound) {
        return nil, nil
    }
    if err != nil {
        return nil, err
    }

    return &user, nil
}
```

---

# 二十四、database/sql 事务

## 【文本块 45】Q：Go 里怎么写事务？

如果使用 database/sql，事务通过 `db.BeginTx` 开启，返回 `*sql.Tx`。

基本流程：

1. BeginTx 开启事务
2. 用 tx 执行所有 SQL
3. 任何一步失败就 Rollback
4. 全部成功就 Commit
5. 注意 Commit 也可能失败

标准写法：

```go
tx, err := db.BeginTx(ctx, nil)
if err != nil {
    return err
}
defer tx.Rollback()

// do something

if err := tx.Commit(); err != nil {
    return err
}
```

注意：defer Rollback 是安全的。
如果 Commit 成功后再 Rollback，通常会返回 sql.ErrTxDone，可以忽略。

面试里可以这样回答：

database/sql 事务要通过 BeginTx 显式开启，事务内所有操作都必须使用 tx，而不能混用 db。通常 defer Rollback 兜底，最后 Commit。事务边界一般放在 service 层，因为 service 才知道一个业务操作需要包含哪些 repository 调用。

---

## 【代码块 26】database/sql 事务示例

```go
package main

import (
    "context"
    "database/sql"
    "fmt"
)

func Transfer(ctx context.Context, db *sql.DB, fromID, toID int64, amount int64) error {
    tx, err := db.BeginTx(ctx, nil)
    if err != nil {
        return err
    }

    defer tx.Rollback()

    result, err := tx.ExecContext(ctx,
        "UPDATE accounts SET balance = balance - ? WHERE id = ? AND balance >= ?",
        amount, fromID, amount,
    )
    if err != nil {
        return err
    }

    rows, err := result.RowsAffected()
    if err != nil {
        return err
    }
    if rows != 1 {
        return fmt.Errorf("insufficient balance")
    }

    _, err = tx.ExecContext(ctx,
        "UPDATE accounts SET balance = balance + ? WHERE id = ?",
        amount, toID,
    )
    if err != nil {
        return err
    }

    if err := tx.Commit(); err != nil {
        return err
    }

    return nil
}
```

---

## 【文本块 46】追问：事务里为什么不能混用 db 和 tx？

因为 db 表示连接池，不代表当前事务连接。

事务 tx 绑定的是某一个数据库连接。事务内的所有 SQL 必须在这个连接上执行，才能属于同一个事务。

如果事务中一部分操作用了 tx，另一部分用了 db，那么 db.Exec 可能从连接池里拿到另一个连接执行，这个操作就不在事务里。

这会导致：

* 回滚不生效
* 数据不一致
* 事务边界失效

所以事务内必须统一使用 tx。

面试里可以这样回答：

事务是绑定在具体连接上的，tx 代表这个事务连接。事务内如果混用 db，就可能走连接池里的其他连接，导致操作不属于当前事务，回滚也回不掉。这是 Go 数据库事务非常常见的坑。

---

# 二十五、GORM 事务

## 【文本块 47】Q：GORM 怎么写事务？

GORM 推荐使用：

```go
db.Transaction(func(tx *gorm.DB) error {
    // 使用 tx 执行业务操作
    return nil
})
```

如果回调返回 error，GORM 会自动 Rollback。
如果返回 nil，GORM 会 Commit。
如果回调里 panic，GORM 也会 Rollback 后继续抛出 panic。

这是最推荐的方式。

---

## 【代码块 27】GORM Transaction 示例

```go
package service

import (
    "context"
    "fmt"

    "gorm.io/gorm"
)

type Account struct {
    ID      uint
    Balance int64
}

func Transfer(ctx context.Context, db *gorm.DB, fromID, toID uint, amount int64) error {
    return db.WithContext(ctx).Transaction(func(tx *gorm.DB) error {
        result := tx.Model(&Account{}).
            Where("id = ? AND balance >= ?", fromID, amount).
            Update("balance", gorm.Expr("balance - ?", amount))

        if result.Error != nil {
            return result.Error
        }

        if result.RowsAffected != 1 {
            return fmt.Errorf("insufficient balance")
        }

        if err := tx.Model(&Account{}).
            Where("id = ?", toID).
            Update("balance", gorm.Expr("balance + ?", amount)).Error; err != nil {
            return err
        }

        return nil
    })
}
```

---

## 【文本块 48】GORM 手动事务写法

也可以手动 Begin、Commit、Rollback：

```go
tx := db.Begin()
if tx.Error != nil {
    return tx.Error
}

defer tx.Rollback()

if err := tx.Create(&order).Error; err != nil {
    return err
}

if err := tx.Commit().Error; err != nil {
    return err
}
```

但更推荐 Transaction 回调，代码更简洁，也更不容易漏 Rollback。

---

## 【代码块 28】GORM 手动事务

```go
func CreateOrder(ctx context.Context, db *gorm.DB, order *Order) error {
    tx := db.WithContext(ctx).Begin()
    if tx.Error != nil {
        return tx.Error
    }

    defer tx.Rollback()

    if err := tx.Create(order).Error; err != nil {
        return err
    }

    if err := tx.Commit().Error; err != nil {
        return err
    }

    return nil
}
```

---

# 二十六、事务边界设计

## 【文本块 49】Q：事务应该放在哪一层？

事务边界通常应该放在 service 层。

原因是：

Handler 只知道 HTTP 参数和响应，不应该关心数据库事务细节。
Repository 只知道单个数据访问操作，不知道完整业务流程。
Service 才知道一个业务动作需要包含哪些数据库操作，以及它们是否要原子提交。

例如“创建订单”可能包括：

1. 创建订单
2. 扣减库存
3. 扣减优惠券
4. 写支付单
5. 写操作日志

这些步骤是否需要一个事务，只有 service 层最清楚。

所以推荐：

```text
Handler
  ↓
Service 开事务
  ↓
Repository 使用 tx 执行 SQL
```

不要让每个 repository 自己随便开事务，否则多个 repo 操作无法合并到一个事务里。

面试里可以这样回答：

事务边界应该放在业务 service 层，因为 service 知道一个业务用例包含哪些数据操作需要保持原子性。Repository 只负责执行具体 SQL，不应该自己决定事务边界。Handler 也不应该直接操作事务，否则 HTTP 层和数据层耦合太重。

---

## 【文本块 50】Repository 怎么兼容普通 db 和 tx？

常见做法是让 repository 接收一个抽象接口，或者在 service 事务中把 tx 传给 repository 方法。

GORM 场景里，`*gorm.DB` 本身可以代表普通 DB，也可以代表事务 tx。

所以 repo 方法可以接收 db 参数：

```go
func (r *OrderRepo) Create(ctx context.Context, db *gorm.DB, order *Order) error
```

或者 repo 内部持有 db，事务场景额外传 tx：

```go
func (r *OrderRepo) WithTx(tx *gorm.DB) *OrderRepo
```

更简单的项目里，也可以在 service 中直接用 tx 调 repo 的内部方法。

关键原则：

> 事务内所有数据库操作必须使用同一个 tx。

---

## 【代码块 29】Service 控制事务，Repo 使用 tx

```go
type OrderRepo struct{}

func (r *OrderRepo) Create(ctx context.Context, db *gorm.DB, order *Order) error {
    return db.WithContext(ctx).Create(order).Error
}

type InventoryRepo struct{}

func (r *InventoryRepo) Deduct(ctx context.Context, db *gorm.DB, productID uint, count int) error {
    result := db.WithContext(ctx).
        Model(&Inventory{}).
        Where("product_id = ? AND stock >= ?", productID, count).
        Update("stock", gorm.Expr("stock - ?", count))

    if result.Error != nil {
        return result.Error
    }
    if result.RowsAffected != 1 {
        return fmt.Errorf("not enough stock")
    }
    return nil
}

type OrderService struct {
    db            *gorm.DB
    orderRepo     *OrderRepo
    inventoryRepo *InventoryRepo
}

func (s *OrderService) CreateOrder(ctx context.Context, order *Order, productID uint, count int) error {
    return s.db.WithContext(ctx).Transaction(func(tx *gorm.DB) error {
        if err := s.inventoryRepo.Deduct(ctx, tx, productID, count); err != nil {
            return err
        }

        if err := s.orderRepo.Create(ctx, tx, order); err != nil {
            return err
        }

        return nil
    })
}
```

---

# 二十七、事务隔离级别与锁

## 【文本块 51】Q：Go 里怎么设置事务隔离级别？

如果使用 database/sql，可以在 BeginTx 时设置：

```go
tx, err := db.BeginTx(ctx, &sql.TxOptions{
    Isolation: sql.LevelRepeatableRead,
    ReadOnly:  false,
})
```

常见隔离级别：

* LevelReadUncommitted
* LevelReadCommitted
* LevelRepeatableRead
* LevelSerializable

具体是否支持取决于数据库。

GORM 如果需要设置隔离级别，可以通过底层 database/sql 或执行 SQL：

```go
db.Exec("SET TRANSACTION ISOLATION LEVEL READ COMMITTED")
```

但实际项目里，更多是依赖数据库默认隔离级别，并在关键位置使用行锁或乐观锁控制并发。

---

## 【代码块 30】database/sql 设置隔离级别

```go
func DoInTx(ctx context.Context, db *sql.DB) error {
    tx, err := db.BeginTx(ctx, &sql.TxOptions{
        Isolation: sql.LevelReadCommitted,
        ReadOnly:  false,
    })
    if err != nil {
        return err
    }

    defer tx.Rollback()

    // do something with tx

    return tx.Commit()
}
```

---

## 【文本块 52】Q：GORM 怎么写行锁？

可以使用 Clauses：

```go
import "gorm.io/gorm/clause"

db.Clauses(clause.Locking{Strength: "UPDATE"}).Find(&users)
```

相当于：

```sql
SELECT * FROM users FOR UPDATE
```

在事务中使用时，可以锁定当前读到的行，防止并发事务修改。

典型场景：

* 扣库存
* 扣余额
* 抢任务
* 状态机流转

但行锁要谨慎：

* 必须在事务中使用才有意义
* 事务不能太长
* 锁顺序要固定，避免死锁
* where 条件要走索引，否则可能锁很多行

---

## 【代码块 31】GORM FOR UPDATE 示例

```go
import (
    "context"

    "gorm.io/gorm"
    "gorm.io/gorm/clause"
)

type Product struct {
    ID    uint
    Stock int
}

func DeductStock(ctx context.Context, db *gorm.DB, productID uint, count int) error {
    return db.WithContext(ctx).Transaction(func(tx *gorm.DB) error {
        var product Product

        if err := tx.Clauses(clause.Locking{Strength: "UPDATE"}).
            Where("id = ?", productID).
            First(&product).Error; err != nil {
            return err
        }

        if product.Stock < count {
            return fmt.Errorf("not enough stock")
        }

        product.Stock -= count

        return tx.Save(&product).Error
    })
}
```

---

## 【文本块 53】追问：扣库存一定要先查再改吗？

不一定。

更推荐的方式往往是单条 SQL 原子扣减：

```sql
UPDATE product
SET stock = stock - ?
WHERE id = ? AND stock >= ?
```

然后检查 affected rows。

这样可以减少事务读写步骤，也能避免一部分并发问题。

GORM 写法：

```go
result := tx.Model(&Product{}).
    Where("id = ? AND stock >= ?", productID, count).
    Update("stock", gorm.Expr("stock - ?", count))
```

如果 RowsAffected 为 0，说明库存不足或商品不存在。

这种写法比“先 select 再 update”更简洁，也更适合高并发扣减。

---

# 二十八、乐观锁与幂等

## 【文本块 54】Q：Go Web 里怎么处理并发更新？

并发更新常见方案：

第一，悲观锁。
使用数据库行锁，比如 `SELECT ... FOR UPDATE`。

适合：

* 冲突频繁
* 强一致要求高
* 临界区逻辑复杂

缺点：

* 锁等待
* 死锁风险
* 吞吐下降

第二，乐观锁。
表里加 version 字段，更新时带上 version 条件。

```sql
UPDATE product
SET stock = ?, version = version + 1
WHERE id = ? AND version = ?
```

如果 affected rows 为 0，说明被别人更新过，需要重试或返回失败。

适合：

* 冲突不高
* 希望减少锁等待
* 状态更新简单

第三，条件更新。
比如扣库存用：

```sql
WHERE stock >= count
```

第四，幂等控制。
对于支付回调、订单创建、消息消费，要用 request_id、order_no、message_id 做幂等。

面试里可以这样回答：

并发更新不是 Go 框架层解决的，本质还是数据库并发控制和业务幂等。强一致场景可以用事务和行锁；冲突不高可以用 version 乐观锁；扣库存这类可以用条件更新；外部请求和消息消费要做幂等键，避免重复执行。

---

## 【代码块 32】乐观锁更新示例

```go
type Product struct {
    ID      uint
    Stock   int
    Version int
}

func UpdateStockWithVersion(ctx context.Context, db *gorm.DB, productID uint, oldVersion int, newStock int) error {
    result := db.WithContext(ctx).
        Model(&Product{}).
        Where("id = ? AND version = ?", productID, oldVersion).
        Updates(map[string]any{
            "stock":   newStock,
            "version": gorm.Expr("version + 1"),
        })

    if result.Error != nil {
        return result.Error
    }

    if result.RowsAffected != 1 {
        return fmt.Errorf("version conflict")
    }

    return nil
}
```

---

# 二十九、统一错误处理

## 【文本块 55】Q：Go Web 项目里怎么统一处理错误？

Go 里不像 Spring 那样用 @ControllerAdvice。

常见做法有两种：

第一，handler 内部显式处理 error。

```go
if err != nil {
    c.JSON(...)
    return
}
```

适合简单项目。

第二，封装统一 handler 返回 error。

例如定义：

```go
type AppHandler func(*gin.Context) error
```

然后 middleware 或 wrapper 统一处理 error。

这样可以让 handler 更干净。

关键点：

* 业务错误转成业务错误码
* 参数错误返回 400
* 鉴权错误返回 401/403
* 资源不存在返回 404
* 系统错误返回 500
* 日志里记录内部 error
* 响应中不要暴露内部细节

---

## 【代码块 33】Gin 统一错误包装

```go
package main

import (
    "errors"
    "net/http"

    "github.com/gin-gonic/gin"
)

var ErrNotFound = errors.New("not found")

type AppHandler func(c *gin.Context) error

func Wrap(h AppHandler) gin.HandlerFunc {
    return func(c *gin.Context) {
        if err := h(c); err != nil {
            switch {
            case errors.Is(err, ErrNotFound):
                c.JSON(http.StatusNotFound, gin.H{
                    "code":    "NOT_FOUND",
                    "message": "resource not found",
                })
            default:
                c.JSON(http.StatusInternalServerError, gin.H{
                    "code":    "INTERNAL_ERROR",
                    "message": "internal server error",
                })
            }
        }
    }
}

func main() {
    r := gin.Default()

    r.GET("/users/:id", Wrap(func(c *gin.Context) error {
        return ErrNotFound
    }))

    r.Run(":8080")
}
```

---

## 【文本块 56】工程建议

不要在 repository 层直接返回 HTTP 状态码。
不要在 service 层依赖 gin.Context。
不要把数据库错误原样返回给前端。

更合理的错误转换链路：

```text
Repository 返回底层 error
   ↓
Service 包装业务上下文
   ↓
Handler 或 error middleware 转成 HTTP 响应
   ↓
日志记录完整错误
   ↓
前端收到稳定错误码和 message
```

---

# 三十、测试与可测试性

## 【文本块 57】Q：Go Web 项目怎么写单元测试？

Go 的工程化设计很强调可测试性。

如果 Handler 直接依赖数据库、Redis、Gin Context，就很难测试。

更好的做法：

* Handler 依赖 service interface
* Service 依赖 repo interface
* 测试时注入 mock/fake
* HTTP 层可以用 httptest
* 数据库层可以用 sqlmock 或测试库
* 事务逻辑可以用集成测试验证

net/http 标准库提供了 httptest，可以模拟请求和响应。

---

## 【代码块 34】net/http handler 测试

```go
package main

import (
    "net/http"
    "net/http/httptest"
    "testing"
)

func hello(w http.ResponseWriter, r *http.Request) {
    w.WriteHeader(http.StatusOK)
    w.Write([]byte("hello"))
}

func TestHello(t *testing.T) {
    req := httptest.NewRequest(http.MethodGet, "/hello", nil)
    rec := httptest.NewRecorder()

    hello(rec, req)

    if rec.Code != http.StatusOK {
        t.Fatalf("expected 200, got %d", rec.Code)
    }

    if rec.Body.String() != "hello" {
        t.Fatalf("unexpected body: %s", rec.Body.String())
    }
}
```

---

## 【文本块 58】追问：为什么 Go 里测试推动 interface 设计？

Go 里 interface 是隐式实现的，不需要实现方显式声明。

这让我们可以在使用方定义最小接口。

例如 service 只需要 repo 的一个方法：

```go
type UserRepo interface {
    FindByID(ctx context.Context, id int64) (*User, error)
}
```

测试时可以定义 fake：

```go
type fakeUserRepo struct{}
```

只要实现 FindByID，就能注入 service。

这比为了“架构完整”提前定义巨大接口更符合 Go 风格。

---

# 三十一、本章高频面试题速记

## 【文本块 59】net/http 速记

net/http 核心是 Handler：

```go
type Handler interface {
    ServeHTTP(ResponseWriter, *Request)
}
```

HandlerFunc 让普通函数也能作为 handler。

Request 表示请求，ResponseWriter 写响应。

Request.Body 是流，通常只能读一次。

ResponseWriter 第一次 Write 会隐式写 200。

生产环境建议显式配置 http.Server：

* ReadHeaderTimeout
* ReadTimeout
* WriteTimeout
* IdleTimeout
* MaxHeaderBytes

HTTP client 要复用，设置 Timeout，关闭 response body，传 context。

优雅停机用 server.Shutdown(ctx)，不是直接 Close。

---

## 【文本块 60】Gin 速记

Gin 是 net/http 的工程化封装。

核心：

* gin.Engine
* gin.Context
* middleware chain
* 路由分组
* 参数绑定
* JSON 响应
* Recovery
* Logger

gin.Context 不是 context.Context。

下游 DB/RPC 应该传：

```go
c.Request.Context()
```

不要把 gin.Context 传到 service/repo 层。

gin.Context 不建议跨 goroutine 使用。异步任务应复制必要数据。

ShouldBindJSON 比 BindJSON 更可控。

---

## 【文本块 61】middleware 速记

middleware 是 Go Web 里实现横切逻辑的主要方式。

常见 middleware：

* recover
* access log
* trace_id
* auth
* CORS
* rate limit
* metrics
* body limit
* timeout

net/http middleware 本质：

```go
func(next http.Handler) http.Handler
```

Gin middleware 本质：

```go
func(c *gin.Context)
```

c.Next 继续执行后续链路。
c.Abort 中断后续链路。

middleware 顺序很重要，recover、trace、log 通常靠外。

---

## 【文本块 62】DI 速记

Go 没有 Spring IOC 那种默认运行时容器。

常见依赖注入方式：

* 手写构造函数
* Google Wire 编译期 DI
* Fx/Dig 运行时 DI

默认推荐构造函数注入。

优点：

* 依赖清晰
* 类型安全
* 少反射
* 方便测试
* 生命周期可控

interface 通常放在使用方，定义最小能力。

不要过度抽象，也不要让 service 依赖 gin.Context。

---

## 【文本块 63】GORM 速记

GORM 是 ORM，database/sql 是底层数据库接口。

`*sql.DB` 是连接池，不是单个连接。

GORM 底层也要配置连接池：

* SetMaxOpenConns
* SetMaxIdleConns
* SetConnMaxLifetime
* SetConnMaxIdleTime

GORM 常见坑：

* Updates struct 不更新零值
* Find 查不到不一定返回 ErrRecordNotFound
* First 查不到会返回 ErrRecordNotFound
* Preload 滥用导致数据量过大
* N+1 查询
* 深分页
* AutoMigrate 不建议生产直接用
* 复杂 SQL 要审查生成 SQL
* DB 操作要 WithContext(ctx)

---

## 【文本块 64】事务速记

database/sql 事务：

```go
tx, err := db.BeginTx(ctx, nil)
defer tx.Rollback()
...
return tx.Commit()
```

GORM 推荐：

```go
db.Transaction(func(tx *gorm.DB) error {
    return nil
})
```

事务边界放在 service 层。

事务内所有操作必须使用同一个 tx，不能混用 db。

事务不要太长，事务里不要做慢 RPC、慢 IO。

扣库存可以用条件更新：

```sql
UPDATE product
SET stock = stock - ?
WHERE id = ? AND stock >= ?
```

并检查 affected rows。

并发控制方案：

* 悲观锁：FOR UPDATE
* 乐观锁：version
* 条件更新
* 幂等 key

---

# 三十二、本章最终面试回答模板

## 【文本块 65】综合回答模板

如果面试官让我整体讲 Go Web 工程化，我会这样回答：

Go Web 的基础是标准库 net/http，它的核心抽象是 Handler 接口。一次请求进来后，net/http 会把请求封装成 *http.Request，把响应写入抽象成 ResponseWriter，然后交给对应的 Handler 处理。Go 的请求处理模型通常是一个请求一个 goroutine，开发者可以用同步代码写 handler，底层 runtime 通过 goroutine 调度和 netpoller 支撑高并发。生产环境里，我不会直接裸用 http.ListenAndServe，而是会显式配置 http.Server 的 ReadHeaderTimeout、ReadTimeout、WriteTimeout、IdleTimeout，并通过 Shutdown 做优雅停机。

Gin 是对 net/http 的工程化封装，它实现了 http.Handler，在标准库基础上提供了路由树、gin.Context、middleware、参数绑定、JSON 响应和 recover 等能力。gin.Context 主要服务于 HTTP 层，不应该传到 service 和 repository 层；下游 DB、Redis、RPC 应该传标准库的 context.Context，也就是 c.Request.Context()。gin.Context 也不建议随意跨 goroutine 使用，如果要异步处理，应该复制必要数据并用 context 控制生命周期。

middleware 是 Go Web 里实现横切逻辑的主要方式，类似 Java 里的 Filter、Interceptor 和一部分 AOP。常见 middleware 包括 recover、access log、trace_id、鉴权、CORS、限流、metrics。net/http middleware 本质是 func(next http.Handler) http.Handler，Gin middleware 则是 func(c *gin.Context)。通过 c.Next 可以继续执行后续链路，通过 c.Abort 可以中断请求。middleware 顺序很重要，recover、trace、log 这类通用能力通常放在比较外层。

Go 没有 Spring IOC 那种运行时大容器，依赖注入通常通过构造函数显式完成。比如先创建 repo，再创建 service，再创建 handler。这样依赖关系清楚、类型安全、少反射，也方便单元测试。项目复杂后可以用 Google Wire 做编译期依赖注入。interface 通常定义在使用方，保持最小能力，不要为了架构感过度抽象。

数据库访问上，Go 标准库 database/sql 提供底层连接池和事务能力，GORM 是基于它之上的 ORM。GORM 提高了 CRUD 和模型映射效率，但不能替代 SQL 思维。生产里要配置连接池参数，比如最大连接数、最大空闲连接数、连接生命周期。GORM 查询要注意 N+1、Preload 滥用、深分页、Updates 零值字段不更新、AutoMigrate 生产风险等问题。所有数据库操作都应该尽量 WithContext(ctx)，让请求取消和超时能传递到数据库层。

事务方面，database/sql 通过 BeginTx 开启事务，GORM 推荐使用 db.Transaction 回调。事务边界应该放在 service 层，因为 service 才知道一个业务用例包含哪些数据库操作需要原子提交。事务内所有操作必须使用同一个 tx，不能混用 db，否则操作可能跑到连接池里的其他连接上，导致不属于当前事务。事务里不要做慢 RPC 或长时间 IO，避免锁持有时间过长。高并发更新可以根据场景选择条件更新、FOR UPDATE 悲观锁、version 乐观锁和幂等 key。

一句话总结：Go Web 工程化的核心不是复刻 Spring，而是用 net/http/Gin 处理请求，用 middleware 管横切逻辑，用显式构造函数做依赖注入，用 GORM 或 database/sql 管数据访问，并在 service 层清晰控制事务和业务边界。
